In [1]:
import sys
print(sys.executable)

/Users/hussaintaheri/Desktop/sports-win-predictor/venv/bin/python


In [2]:
# we have doubled our data size by pulling new data lets see how our model will perform on the dataset and well also be using the
# hyperparameters we found using the bayesian optimization from optuna
import pandas as pd

# get our feature engineered datasets
df1 = pd.read_csv("../data/nba_pbp_2020_2025_fe.csv")
df2 = pd.read_csv("../data/nba_pbp_2014_2020_fe.csv")

# see the shapes of our data
df1.shape, df2.shape

((3282177, 20), (3368298, 20))

In [3]:
# see if both of our dataframes matches
df1.head()

,Unnamed: 0,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,0,1,0.0,0.0,0.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1,1,0.0,0.0,2.0,1,17,0,0,0,0,0,0,1,0,0,0,0,0,0
2,2,1,0.0,0.0,1.0,1,20,0,0,0,0,0,0,0,1,0,0,0,0,0
3,3,1,0.0,0.0,1.0,1,37,0,0,0,0,0,0,1,0,0,0,0,0,0
4,4,1,0.0,0.0,2.0,1,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [4]:
df2.head()

,Unnamed: 0,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,0,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1,1,0.0,0.0,2.0,0,4,0,0,0,0,0,0,0,0,0,0,0,1,0
2,2,1,0.0,0.0,1.0,0,12,0,0,0,0,0,0,1,0,0,0,0,0,0
3,3,1,0.0,0.0,2.0,0,15,0,0,0,0,0,0,0,1,0,0,0,0,0
4,4,1,0.0,0.0,2.0,0,21,0,0,0,0,0,0,1,0,0,0,0,0,0


In [5]:
# drop the unnamed column
df1 = df1.drop("Unnamed: 0", axis = 1)
df2 = df2.drop("Unnamed: 0", axis = 1)

In [6]:
# check null values
df1.isna().sum()

period                       0
scoreHome                    0
scoreAway                    0
location                     0
home_team_won                0
seconds                      0
actionType_Ejection          0
actionType_Foul              0
actionType_Free Throw        0
actionType_Instant Replay    0
actionType_Jump Ball         0
actionType_Made Shot         0
actionType_Missed Shot       0
actionType_Rebound           0
actionType_Substitution      0
actionType_Timeout           0
actionType_Turnover          0
actionType_Violation         0
actionType_period            0
dtype: int64

In [7]:
df2.isna().sum()

period                       0
scoreHome                    0
scoreAway                    0
location                     0
home_team_won                0
seconds                      0
actionType_Ejection          0
actionType_Foul              0
actionType_Free Throw        0
actionType_Instant Replay    0
actionType_Jump Ball         0
actionType_Made Shot         0
actionType_Missed Shot       0
actionType_Rebound           0
actionType_Substitution      0
actionType_Timeout           0
actionType_Turnover          0
actionType_Violation         0
actionType_period            0
dtype: int64

In [8]:
# check datatypes
df1.dtypes

period                         int64
scoreHome                    float64
scoreAway                    float64
location                     float64
home_team_won                  int64
seconds                        int64
actionType_Ejection            int64
actionType_Foul                int64
actionType_Free Throw          int64
actionType_Instant Replay      int64
actionType_Jump Ball           int64
actionType_Made Shot           int64
actionType_Missed Shot         int64
actionType_Rebound             int64
actionType_Substitution        int64
actionType_Timeout             int64
actionType_Turnover            int64
actionType_Violation           int64
actionType_period              int64
dtype: object

In [9]:
df2.dtypes

period                         int64
scoreHome                    float64
scoreAway                    float64
location                     float64
home_team_won                  int64
seconds                        int64
actionType_Ejection            int64
actionType_Foul                int64
actionType_Free Throw          int64
actionType_Instant Replay      int64
actionType_Jump Ball           int64
actionType_Made Shot           int64
actionType_Missed Shot         int64
actionType_Rebound             int64
actionType_Substitution        int64
actionType_Timeout             int64
actionType_Turnover            int64
actionType_Violation           int64
actionType_period              int64
dtype: object

In [10]:
# concact dataframes into one, and reset the indexs
df = pd.concat([df1, df2], ignore_index = True)
df.shape

(6650475, 19)

In [11]:
# we now have over 6,600,000 rows of data
df.head()
df.to_csv("../data/nba_pbp_2014_2025_fe.csv")

In [12]:
# lets find where we have around 80 percent of data where a new game starts to create our features and targets
df.iloc[5320037:5320040]

,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
5320037,4,110.0,102.0,0.0,1,2880,0,0,0,0,0,0,0,0,0,0,0,0,1
5320038,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
5320039,1,0.0,0.0,2.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [13]:
# now that we found the row where a new game starts at around 80 percent data
# we can create our features and targets 
X = df.drop("home_team_won", axis = 1)
y = df["home_team_won"]

X_train, X_test, y_train, y_test = X.iloc[:5320038], X.iloc[5320038:], y.iloc[:5320038], y.iloc[5320038:]

In [14]:
X_train.tail()

,period,scoreHome,scoreAway,location,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
5320033,4,110.0,99.0,2.0,2841,0,0,0,0,0,0,0,1,0,0,0,0,0
5320034,4,110.0,99.0,2.0,2862,0,0,0,0,0,0,0,0,0,0,1,0,0
5320035,4,110.0,99.0,1.0,2862,0,0,0,0,0,0,0,0,0,0,0,0,0
5320036,4,110.0,102.0,1.0,2870,0,0,0,0,0,1,0,0,0,0,0,0,0
5320037,4,110.0,102.0,0.0,2880,0,0,0,0,0,0,0,0,0,0,0,0,1


In [15]:
X_test.head()

,period,scoreHome,scoreAway,location,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
5320038,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
5320039,1,0.0,0.0,2.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
5320040,1,0.0,0.0,2.0,14,0,0,0,0,0,0,1,0,0,0,0,0,0
5320041,1,0.0,0.0,1.0,16,0,0,0,0,0,0,0,1,0,0,0,0,0
5320042,1,0.0,0.0,1.0,31,0,0,0,0,0,0,0,0,0,0,1,0,0


In [16]:
# check if the data is the same for the training and test set
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5320038, 18), (1330437, 18), (5320038,), (1330437,))

In [17]:
%%time
# import the model we want to first use
from xgboost import XGBClassifier

# import the metrics we wanna use
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

# track the models we want to use on our data set
import mlflow

# start an mlflow run
with mlflow.start_run():
    
    # initialize the model and use the best parameters we got and unpack them using **
    model = XGBClassifier()
    
    # train the model
    xg = model.fit(X_train, y_train)

    # log the parameters in this case the default ones
    mlflow.log_params(model.get_params())

    # log the model and give it a name
    model_info = mlflow.xgboost.log_model(xg, name = "2014_2020_xgboost_model")
    
    # get the model predicted values
    y_pred = model.predict(X_test)

    # get the probabilite values, so we can use for log loss and roc_auc
    y_proba = model.predict_proba(X_test)

    # get the models metrics
    score = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba[:, 1])
    roc_auc = roc_auc_score(y_test, y_proba[:, 1])

    # log the metrics
    mlflow.log_metrics({"log_loss": ll, "accuracy_score": score, "roc_auc": roc_auc})

/Users/hussaintaheri/Desktop/sports-win-predictor/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: user 30.2 s, sys: 1.44 s, total: 31.6 s
Wall time: 13.4 s
